In [7]:
# ============================================================
# QCTrojan-Bench: Quantum Circuit Trojan Detection Benchmark
# ============================================================
# Notebook:  S3_feature_extraction.ipynb
# Purpose:   Extract normalised structural features from all
#            9,000 circuits (benign + tampered) across three
#            circuit representations:
#              1. Logical (as stored)
#              2. Transpiled opt_level=0
#              3. Transpiled opt_level=3
# Authors:   Zeeshan Ajmal
#            University of Oulu, Finland
# Version:   QCTrojan-Bench v1.0
# License:   CC BY 4.0
# ============================================================
#
# CELL 1 — Imports, Configuration, Backend Setup
#
# Prerequisites:
#   S1_benign_circuits.ipynb complete (3,000 benign circuits)
#   S2_tampered_circuits.ipynb complete (6,000 tampered circuits)
# ============================================================

import os
import json
import csv
import math
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean, pstdev
from collections import defaultdict

from qiskit import QuantumCircuit, qpy, transpile
from qiskit.converters import circuit_to_dag
from qiskit_ibm_runtime.fake_provider import FakeGuadalupeV2

# ── Versioning ───────────────────────────────────────────────
QISKIT_VERSION     = "2.3.1"
EXTRACTOR_VERSION  = "QCTrojan-Bench-feature-extractor-v1.0"
DATASET_VERSION    = "v1"
FEATURES_VERSION   = "v1"

# ── Paths (auto-detected) ─────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATASET_ROOT = PROJECT_ROOT / "dataset"

BENIGN_DIR   = DATASET_ROOT / "circuits" / "benign"
TAMPERED_DIR = DATASET_ROOT / "circuits" / "tampered"
FEATURES_DIR = DATASET_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset constants ─────────────────────────────────────────
FAMILIES = [
    "deutsch_jozsa",
    "grover",
    "qaoa",
    "vqc",
    "qft",
]
TROJAN_TYPES = ["static", "triggered"]

# ── Transpilation configuration (FROZEN) ─────────────────────
# These values must match what is reported in the paper.
# Do NOT change after feature extraction begins.
BACKEND          = FakeGuadalupeV2()
BACKEND_NAME     = "FakeGuadalupeV2"
BACKEND_NQUBITS  = BACKEND.configuration().n_qubits
TRANSPILER_SEED  = 12345
OPT_LEVELS       = [0, 3]

# ── Verify backend ────────────────────────────────────────────
assert BACKEND_NQUBITS >= 16, \
    f"Backend too small: {BACKEND_NQUBITS} qubits"

# ── Transpilation helper ──────────────────────────────────────
def transpile_circuit(qc: QuantumCircuit,
                      opt_level: int) -> QuantumCircuit:
    """
    Transpile a circuit deterministically.

    Uses FakeGuadalupeV2 backend with fixed seed for
    reproducible layout, routing, and optimisation.

    Args:
        qc:        logical circuit (no measurements)
        opt_level: 0 (no optimisation) or 3 (maximum)

    Returns:
        Transpiled QuantumCircuit
    """
    return transpile(
        qc,
        backend=BACKEND,
        optimization_level=opt_level,
        seed_transpiler=TRANSPILER_SEED,
    )

# ── Summary ───────────────────────────────────────────────────
print("=" * 55)
print("QCTrojan-Bench — S3 Feature Extraction")
print("=" * 55)
print(f"  Extractor version : {EXTRACTOR_VERSION}")
print(f"  Qiskit version    : {QISKIT_VERSION}")
print(f"  Backend           : {BACKEND_NAME} "
      f"({BACKEND_NQUBITS} qubits)")
print(f"  Transpiler seed   : {TRANSPILER_SEED}")
print(f"  Opt levels        : {OPT_LEVELS}")
print(f"  Features dir      : {FEATURES_DIR}")
print("=" * 55)
print("Cell 1 complete. Backend ready.")


C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\.venv\Lib\site-packages\samplomatic\__init__.py:20: UserWarning: 
You have imported samplomatic==0.17.1 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


QCTrojan-Bench — S3 Feature Extraction
  Extractor version : QCTrojan-Bench-feature-extractor-v1.0
  Qiskit version    : 2.3.1
  Backend           : FakeGuadalupeV2 (16 qubits)
  Transpiler seed   : 12345
  Opt levels        : [0, 3]
  Features dir      : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\features
Cell 1 complete. Backend ready.


In [8]:
# ============================================================
# CELL 2 — Feature Extraction Functions
# ============================================================
#
# Defines all feature extraction functions used for every
# circuit representation (logical, opt0, opt3).
#
# Feature groups:
#
# Group 1 — Normalised structural ratios
#   depth_per_qubit         : circuit depth / n_qubits
#   twoq_per_depth          : two-qubit gates / depth
#   cx_ratio                : CX gates / total two-qubit gates
#
# Group 2 — Interaction topology
#   max_degree_norm         : max qubit degree / (n_qubits-1)
#   degree_std_over_mean    : std(degrees) / mean(degrees)
#   max_edge_weight_over_total : max edge weight / total weight
#   frac_interactions_top1  : weight of top qubit / total
#   frac_interactions_top2  : weight of top 2 qubits / total
#
# Group 3 — Layer locality
#   entangling_layers_ratio : layers with 2q gate / total layers
#   fraction_layers_active  : same as above (alias)
#   max_twoq_in_layer_norm  : max 2q in layer / floor(n/2)
#   layer_twoq_std_over_mean: std / mean of 2q per layer
#
# Group 4 — Optimisation sensitivity (opt3 vs opt0)
#   opt_depth_ratio         : depth(opt3) / depth(opt0)
#   opt_twoq_ratio          : twoq(opt3) / twoq(opt0)
#   opt_interaction_overlap : Jaccard overlap of interaction
#                             graphs between opt0 and opt3
#
# Rules:
#   - No raw absolute counts used as features
#   - All features are normalised ratios or topology statistics
#   - n_qubits stored as metadata only
#   - Features extracted independently per representation
# ============================================================

def safe_div(a: float, b: float,
             default: float = 0.0) -> float:
    """Division that returns default when denominator is zero."""
    return a / b if b > 1e-12 else default


def extract_structural_ratios(qc: QuantumCircuit) -> dict:
    """
    Group 1: Normalised structural ratios.

    All features are ratios — no raw counts.
    """
    n      = qc.num_qubits
    depth  = qc.depth()

    total_ops = 0
    num_twoq  = 0
    num_cx    = 0

    for inst in qc.data:
        total_ops += 1
        nq = len(inst.qubits)
        if nq == 2:
            num_twoq += 1
            if inst.operation.name == "cx":
                num_cx += 1

    return {
        "depth_per_qubit": safe_div(depth, n),
        "twoq_per_depth":  safe_div(num_twoq, depth),
        "cx_ratio":        safe_div(num_cx, num_twoq),
    }


def extract_interaction_topology(qc: QuantumCircuit) -> dict:
    """
    Group 2: Qubit interaction topology features.

    Builds a qubit interaction graph from two-qubit gates.
    Nodes = qubits. Edge weight = number of two-qubit gates
    between that qubit pair.
    """
    n = qc.num_qubits

    # Build edge weight dict
    edge_weights = defaultdict(int)
    qubit_degree = defaultdict(int)

    for inst in qc.data:
        if len(inst.qubits) == 2:
            q0 = inst.qubits[0]._index
            q1 = inst.qubits[1]._index
            edge = (min(q0, q1), max(q0, q1))
            edge_weights[edge] += 1
            qubit_degree[q0]   += 1
            qubit_degree[q1]   += 1

    # Total interaction weight
    total_weight = sum(edge_weights.values())

    # Degree statistics
    degrees = [qubit_degree.get(q, 0) for q in range(n)]
    max_deg = max(degrees) if degrees else 0
    mean_deg = mean(degrees) if degrees else 0.0
    std_deg  = pstdev(degrees) if len(degrees) >= 2 else 0.0

    # Edge weight statistics
    weights_sorted = sorted(edge_weights.values(), reverse=True)
    top1_weight = weights_sorted[0] if weights_sorted else 0
    top2_weight = sum(weights_sorted[:2]) if weights_sorted else 0

    return {
        "max_degree_norm":
            safe_div(max_deg, n - 1),
        "degree_std_over_mean":
            safe_div(std_deg, mean_deg),
        "max_edge_weight_over_total":
            safe_div(top1_weight, total_weight),
        "frac_interactions_top1":
            safe_div(top1_weight, total_weight),
        "frac_interactions_top2":
            safe_div(top2_weight, total_weight),
    }


def extract_layer_locality(qc: QuantumCircuit) -> dict:
    """
    Group 3: Layer locality and entanglement concentration.

    Uses DAG layer decomposition to count two-qubit gates
    per layer and compute distribution statistics.
    """
    n   = qc.num_qubits
    dag = circuit_to_dag(qc)

    twoq_per_layer = []
    for layer in dag.layers():
        twoq = sum(
            1 for node in layer["graph"].op_nodes()
            if len(node.qargs) == 2
        )
        twoq_per_layer.append(twoq)

    total_layers  = len(twoq_per_layer) if twoq_per_layer else 1
    active_layers = sum(1 for x in twoq_per_layer if x > 0)
    max_possible  = max(1, n // 2)
    max_twoq      = max(twoq_per_layer) if twoq_per_layer else 0
    m   = mean(twoq_per_layer)   if twoq_per_layer else 0.0
    sd  = pstdev(twoq_per_layer) if len(twoq_per_layer) >= 2 \
          else 0.0

    return {
        "entangling_layers_ratio":
            safe_div(active_layers, total_layers),
        "fraction_layers_active":
            safe_div(active_layers, total_layers),
        "max_twoq_in_layer_norm":
            safe_div(max_twoq, max_possible),
        "layer_twoq_std_over_mean":
            safe_div(sd, m),
    }


def extract_all_features(qc: QuantumCircuit) -> dict:
    """
    Extract all features from a single circuit representation.

    Combines Groups 1, 2, and 3.
    Returns a flat dict of feature_name → float.
    """
    features = {}
    features.update(extract_structural_ratios(qc))
    features.update(extract_interaction_topology(qc))
    features.update(extract_layer_locality(qc))
    return features


def extract_optimisation_sensitivity(
    qc_opt0: QuantumCircuit,
    qc_opt3: QuantumCircuit,
) -> dict:
    """
    Group 4: Optimisation sensitivity features.

    Compares structural properties between opt_level=0
    and opt_level=3 representations of the same circuit.

    In Qiskit, opt_level=0 applies no optimisation passes
    (minimum compilation effort) while opt_level=3 applies
    the most aggressive available optimisation strategy.
    These represent the minimum and maximum compilation
    effort within the Qiskit framework.
    """
    depth0 = qc_opt0.depth()
    depth3 = qc_opt3.depth()

    twoq0 = sum(1 for inst in qc_opt0.data
                if len(inst.qubits) == 2)
    twoq3 = sum(1 for inst in qc_opt3.data
                if len(inst.qubits) == 2)

    # Interaction graph overlap (Jaccard similarity)
    def interaction_edges(qc):
        edges = set()
        for inst in qc.data:
            if len(inst.qubits) == 2:
                q0 = inst.qubits[0]._index
                q1 = inst.qubits[1]._index
                edges.add((min(q0, q1), max(q0, q1)))
        return edges

    edges0 = interaction_edges(qc_opt0)
    edges3 = interaction_edges(qc_opt3)

    intersection = len(edges0 & edges3)
    union        = len(edges0 | edges3)

    return {
        "opt_depth_ratio_opt3_over_opt0":
            safe_div(depth3, depth0),
        "opt_twoq_ratio_opt3_over_opt0":
            safe_div(twoq3, twoq0),
        "opt_interaction_overlap_opt0_vs_opt3":
            safe_div(intersection, union),
    }


def prefix_features(features: dict,
                    prefix: str) -> dict:
    """Add a prefix to all feature keys."""
    return {f"{prefix}_{k}": v for k, v in features.items()}


# ── Smoke test ────────────────────────────────────────────────

from qiskit.circuit.library import n_local

_test_qc = QuantumCircuit(5)
_test_qc.h(range(5))
_test_qc.cx(0, 1)
_test_qc.cx(1, 2)
_test_qc.cx(2, 3)
_test_qc.rz(0.5, 2)
_test_qc.cx(3, 4)

_feats = extract_all_features(_test_qc)

print("Feature extraction smoke test passed.")
print(f"  Features extracted : {len(_feats)}")
print(f"  Feature names      : {list(_feats.keys())}")
print()

_opt0 = transpile_circuit(_test_qc, 0)
_opt3 = transpile_circuit(_test_qc, 3)
_sens = extract_optimisation_sensitivity(_opt0, _opt3)

print(f"  Optimisation sensitivity features:")
for k, v in _sens.items():
    print(f"    {k}: {v:.4f}")

print("\nCell 2 complete. All feature functions ready.")


Feature extraction smoke test passed.
  Features extracted : 12
  Feature names      : ['depth_per_qubit', 'twoq_per_depth', 'cx_ratio', 'max_degree_norm', 'degree_std_over_mean', 'max_edge_weight_over_total', 'frac_interactions_top1', 'frac_interactions_top2', 'entangling_layers_ratio', 'fraction_layers_active', 'max_twoq_in_layer_norm', 'layer_twoq_std_over_mean']

  Optimisation sensitivity features:
    opt_depth_ratio_opt3_over_opt0: 0.5833
    opt_twoq_ratio_opt3_over_opt0: 0.4000
    opt_interaction_overlap_opt0_vs_opt3: 0.3333

Cell 2 complete. All feature functions ready.


In [9]:
# ============================================================
# CELL 3 — Load and Transpile All Circuits
# ============================================================
#
# Loads all 9,000 circuits (3,000 benign + 6,000 tampered)
# and generates three representations per circuit:
#   1. logical  — as stored (no transpilation)
#   2. opt0     — transpiled with opt_level=0
#   3. opt3     — transpiled with opt_level=3
#
# All representations kept in memory for Cell 4.
# Progress is printed every 500 circuits.
#
# Expected time: 10-20 minutes depending on hardware.
# ============================================================

all_records = []   # list of dicts, one per circuit
errors      = []   # any circuits that fail to load/transpile
processed   = 0

print("Loading and transpiling all circuits...")
print("(This takes 10-20 minutes — progress printed every 500)")
print("=" * 55)

# ── Helper: load one family split ────────────────────────────

def load_family_split(
    circuit_dir: Path,
    metadata_dir: Path,
    label: str,
    trojan_type: str,
    family: str,
) -> list:
    """
    Load all circuits from one family/type directory.
    Returns list of record dicts with circuit objects.
    """
    records = []
    qpy_files = sorted(circuit_dir.glob("*.qpy"))

    for qpy_file in qpy_files:
        sample_id = qpy_file.stem
        meta_path = metadata_dir / f"{sample_id}.json"

        # Load circuit
        try:
            with open(qpy_file, "rb") as f:
                qc_logical = qpy.load(f)[0]
        except Exception as e:
            errors.append(f"Load failed {sample_id}: {e}")
            continue

        # Check backend can support this circuit
        if qc_logical.num_qubits > BACKEND_NQUBITS:
            errors.append(
                f"{sample_id}: {qc_logical.num_qubits} qubits "
                f"exceeds backend ({BACKEND_NQUBITS})"
            )
            continue

        # Load metadata
        try:
            with open(meta_path) as f:
                meta = json.load(f)
        except Exception as e:
            errors.append(f"Metadata load failed {sample_id}: {e}")
            continue

        records.append({
            "sample_id":        sample_id,
            "label":            label,
            "trojan_type":      trojan_type,
            "algorithm_family": family,
            "parent_sample_id": meta.get("parent_sample_id"),
            "n_qubits":         qc_logical.num_qubits,
            "qc_logical":       qc_logical,
            "metadata":         meta,
        })

    return records


# ── Load benign circuits ──────────────────────────────────────

print("\nLoading benign circuits...")
for family in FAMILIES:
    circuit_dir  = BENIGN_DIR / family / DATASET_VERSION / "circuits"
    metadata_dir = BENIGN_DIR / family / DATASET_VERSION / "metadata"

    records = load_family_split(
        circuit_dir, metadata_dir,
        label="benign",
        trojan_type="benign",
        family=family,
    )
    all_records.extend(records)
    print(f"  {family:20s} : {len(records)} benign circuits loaded")

# ── Load tampered circuits ────────────────────────────────────

print("\nLoading tampered circuits...")
for family in FAMILIES:
    for trojan_type in TROJAN_TYPES:
        circuit_dir  = (TAMPERED_DIR / family / DATASET_VERSION
                        / trojan_type / "circuits")
        metadata_dir = (TAMPERED_DIR / family / DATASET_VERSION
                        / trojan_type / "metadata")

        records = load_family_split(
            circuit_dir, metadata_dir,
            label="tampered",
            trojan_type=trojan_type,
            family=family,
        )
        all_records.extend(records)
        print(f"  {family:20s} / {trojan_type:10s} : "
              f"{len(records)} circuits loaded")

print(f"\nTotal circuits loaded : {len(all_records)}")
print(f"Load errors           : {len(errors)}")

if errors:
    print("ERRORS:")
    for e in errors[:10]:
        print(f"  {e}")
    raise RuntimeError(
        f"{len(errors)} circuits failed to load. Fix before continuing."
    )

assert len(all_records) == 9000, \
    f"Expected 9000 circuits, got {len(all_records)}"

# ── Transpile all circuits ────────────────────────────────────

print("\nTranspiling all circuits (opt0 and opt3)...")
print("Progress printed every 500 circuits.")
print("-" * 55)

transpile_errors = []

for i, record in enumerate(all_records):
    qc = record["qc_logical"]

    try:
        record["qc_opt0"] = transpile_circuit(qc, 0)
        record["qc_opt3"] = transpile_circuit(qc, 3)
    except Exception as e:
        transpile_errors.append(
            f"Transpile failed {record['sample_id']}: {e}"
        )
        record["qc_opt0"] = None
        record["qc_opt3"] = None

    if (i + 1) % 500 == 0:
        print(f"  [{i + 1:5d} / 9000] "
              f"errors so far: {len(transpile_errors)}")

print("-" * 55)
print(f"Transpilation complete.")
print(f"  Total processed   : {len(all_records)}")
print(f"  Transpile errors  : {len(transpile_errors)}")

if transpile_errors:
    print("TRANSPILE ERRORS:")
    for e in transpile_errors[:10]:
        print(f"  {e}")
    raise RuntimeError(
        f"{len(transpile_errors)} circuits failed transpilation."
    )

# ── Quick sanity check ────────────────────────────────────────

sample = all_records[0]
print(f"\nSanity check on first record:")
print(f"  sample_id  : {sample['sample_id']}")
print(f"  family     : {sample['algorithm_family']}")
print(f"  trojan     : {sample['trojan_type']}")
print(f"  n_qubits   : {sample['n_qubits']}")
print(f"  logical depth : {sample['qc_logical'].depth()}")
print(f"  opt0 depth    : {sample['qc_opt0'].depth()}")
print(f"  opt3 depth    : {sample['qc_opt3'].depth()}")

print("\nCell 3 complete. All circuits loaded and transpiled.")
print("Ready for Cell 4 — feature extraction.")


Loading and transpiling all circuits...
(This takes 10-20 minutes — progress printed every 500)

Loading benign circuits...
  deutsch_jozsa        : 600 benign circuits loaded
  grover               : 600 benign circuits loaded
  qaoa                 : 600 benign circuits loaded
  vqc                  : 600 benign circuits loaded
  qft                  : 600 benign circuits loaded

Loading tampered circuits...
  deutsch_jozsa        / static     : 600 circuits loaded
  deutsch_jozsa        / triggered  : 600 circuits loaded
  grover               / static     : 600 circuits loaded
  grover               / triggered  : 600 circuits loaded
  qaoa                 / static     : 600 circuits loaded
  qaoa                 / triggered  : 600 circuits loaded
  vqc                  / static     : 600 circuits loaded
  vqc                  / triggered  : 600 circuits loaded
  qft                  / static     : 600 circuits loaded
  qft                  / triggered  : 600 circuits loaded

Total

In [10]:
# ============================================================
# CELL 4 — Extract Features and Build CSV
# ============================================================
#
# Extracts all features from three representations of each
# circuit and writes features_v2.csv.
#
# One row per circuit. Columns:
#   Metadata : sample_id, parent_sample_id, label,
#              trojan_type, algorithm_family, n_qubits
#   Features : logical_*, opt0_*, opt3_*, opt_*
#
# Rules:
#   - No raw absolute counts as features
#   - n_qubits is metadata only — NOT a feature column
#   - All feature values are normalised ratios or statistics
#   - Progress printed every 1000 circuits
# ============================================================

print("Extracting features from all 9,000 circuits...")
print("=" * 55)

feature_rows   = []
extract_errors = []

for i, record in enumerate(all_records):

    sample_id  = record["sample_id"]
    qc_logical = record["qc_logical"]
    qc_opt0    = record["qc_opt0"]
    qc_opt3    = record["qc_opt3"]

    try:
        # ── Extract per-representation features ──────────────
        logical_feats = prefix_features(
            extract_all_features(qc_logical), "logical"
        )
        opt0_feats = prefix_features(
            extract_all_features(qc_opt0), "opt0"
        )
        opt3_feats = prefix_features(
            extract_all_features(qc_opt3), "opt3"
        )

        # ── Extract optimisation sensitivity features ─────────
        opt_sensitivity = extract_optimisation_sensitivity(
            qc_opt0, qc_opt3
        )

        # ── Build row ─────────────────────────────────────────
        row = {
            # Metadata (not used as features in ML)
            "sample_id":          sample_id,
            "parent_sample_id":   record["parent_sample_id"],
            "label":              record["label"],
            "trojan_type":        record["trojan_type"],
            "algorithm_family":   record["algorithm_family"],
            "n_qubits":           record["n_qubits"],
        }

        # Features
        row.update(logical_feats)
        row.update(opt0_feats)
        row.update(opt3_feats)
        row.update(opt_sensitivity)

        feature_rows.append(row)

    except Exception as e:
        extract_errors.append(
            f"Feature extraction failed {sample_id}: {e}"
        )
        continue

    if (i + 1) % 1000 == 0:
        print(f"  [{i + 1:5d} / 9000] "
              f"errors so far: {len(extract_errors)}")

print("-" * 55)
print(f"Feature extraction complete.")
print(f"  Rows extracted : {len(feature_rows)}")
print(f"  Errors         : {len(extract_errors)}")

if extract_errors:
    print("ERRORS:")
    for e in extract_errors[:10]:
        print(f"  {e}")
    raise RuntimeError(
        f"{len(extract_errors)} circuits failed feature extraction."
    )

assert len(feature_rows) == 9000, \
    f"Expected 9000 rows, got {len(feature_rows)}"

# ── Determine column order ────────────────────────────────────

META_COLS = [
    "sample_id",
    "parent_sample_id",
    "label",
    "trojan_type",
    "algorithm_family",
    "n_qubits",
]

# Feature columns in consistent order
FEATURE_COLS = [
    k for k in feature_rows[0].keys()
    if k not in META_COLS
]

ALL_COLS = META_COLS + FEATURE_COLS

print(f"\n  Metadata columns : {len(META_COLS)}")
print(f"  Feature columns  : {len(FEATURE_COLS)}")
print(f"  Total columns    : {len(ALL_COLS)}")
print(f"\n  Feature columns:")
for col in FEATURE_COLS:
    print(f"    {col}")

# ── Write CSV ─────────────────────────────────────────────────

CSV_PATH = FEATURES_DIR / f"features_{FEATURES_VERSION}.csv"

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=ALL_COLS)
    writer.writeheader()
    writer.writerows(feature_rows)

print(f"\n  Saved: {CSV_PATH}")

# ── Quick NaN / infinity check ────────────────────────────────

import math

nan_count = 0
inf_count = 0

for row in feature_rows:
    for col in FEATURE_COLS:
        val = row[col]
        if val is None or (isinstance(val, float)
                           and math.isnan(val)):
            nan_count += 1
        if isinstance(val, float) and math.isinf(val):
            inf_count += 1

print(f"\n  NaN values  : {nan_count}  (should be 0)")
print(f"  Inf values  : {inf_count}  (should be 0)")

if nan_count > 0 or inf_count > 0:
    raise RuntimeError(
        "NaN or Inf values detected in feature matrix. "
        "Check safe_div calls in Cell 2."
    )

# ── Label distribution check ──────────────────────────────────

from collections import Counter

label_counts  = Counter(r["label"]       for r in feature_rows)
trojan_counts = Counter(r["trojan_type"] for r in feature_rows)
family_counts = Counter(r["algorithm_family"]
                        for r in feature_rows)

print(f"\n  Label distribution:")
for k, v in sorted(label_counts.items()):
    print(f"    {k:12s} : {v}")

print(f"\n  Trojan type distribution:")
for k, v in sorted(trojan_counts.items()):
    print(f"    {k:12s} : {v}")

print(f"\n  Family distribution:")
for k, v in sorted(family_counts.items()):
    print(f"    {k:20s} : {v}")

print("\nCell 4 complete. features_v1.csv saved.")


Extracting features from all 9,000 circuits...
  [ 1000 / 9000] errors so far: 0
  [ 2000 / 9000] errors so far: 0
  [ 3000 / 9000] errors so far: 0
  [ 4000 / 9000] errors so far: 0
  [ 5000 / 9000] errors so far: 0
  [ 6000 / 9000] errors so far: 0
  [ 7000 / 9000] errors so far: 0
  [ 8000 / 9000] errors so far: 0
  [ 9000 / 9000] errors so far: 0
-------------------------------------------------------
Feature extraction complete.
  Rows extracted : 9000
  Errors         : 0

  Metadata columns : 6
  Feature columns  : 39
  Total columns    : 45

  Feature columns:
    logical_depth_per_qubit
    logical_twoq_per_depth
    logical_cx_ratio
    logical_max_degree_norm
    logical_degree_std_over_mean
    logical_max_edge_weight_over_total
    logical_frac_interactions_top1
    logical_frac_interactions_top2
    logical_entangling_layers_ratio
    logical_fraction_layers_active
    logical_max_twoq_in_layer_norm
    logical_layer_twoq_std_over_mean
    opt0_depth_per_qubit
    opt0_tw

In [12]:
# ============================================================
# CELL 5 — Validation and Extractor Manifest
# ============================================================
#
# Final validation of features_v2.csv and saves the
# extractor manifest for reproducibility.
#
# Validation checks:
#   - Correct row count (9,000)
#   - All feature columns present
#   - No NaN or Inf values
#   - Feature value ranges are plausible (0 to reasonable max)
#   - Label and family distributions correct
#   - parent_sample_id links are valid
#   - Features differ between benign and tampered
#     (basic separability sanity check)
#
# Manifest saved:
#   dataset/features/extractor_manifest.json
# ============================================================

import csv as csv_module

print("=" * 55)
print("S3 — Final Validation")
print("=" * 55)

# ── Reload CSV from disk ──────────────────────────────────────
# Validates the saved file, not just the in-memory data

print("\nReloading features_v1.csv from disk...")
CSV_PATH = FEATURES_DIR / f"features_{FEATURES_VERSION}.csv"

loaded_rows = []
with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv_module.DictReader(f)
    for row in reader:
        loaded_rows.append(row)

print(f"  Rows loaded    : {len(loaded_rows)}")
print(f"  Columns        : {len(loaded_rows[0])}")

errors = []

# ── Row count ─────────────────────────────────────────────────
if len(loaded_rows) != 9000:
    errors.append(
        f"Row count: expected 9000, got {len(loaded_rows)}"
    )

# ── Column presence ───────────────────────────────────────────
loaded_cols = list(loaded_rows[0].keys())
missing_cols = [c for c in ALL_COLS if c not in loaded_cols]
if missing_cols:
    errors.append(f"Missing columns: {missing_cols}")

# ── NaN / Inf / empty check ───────────────────────────────────
nan_count = 0
inf_count = 0
empty_count = 0

for row in loaded_rows:
    for col in FEATURE_COLS:
        val = row.get(col, "")
        if val == "" or val is None:
            empty_count += 1
            continue
        try:
            fval = float(val)
            if math.isnan(fval):
                nan_count += 1
            if math.isinf(fval):
                inf_count += 1
        except ValueError:
            errors.append(
                f"Non-numeric value in {col}: '{val}'"
            )

print(f"\n  NaN count   : {nan_count}  (should be 0)")
print(f"  Inf count   : {inf_count}  (should be 0)")
print(f"  Empty count : {empty_count}  (should be 0)")

if nan_count > 0:
    errors.append(f"{nan_count} NaN values in feature matrix")
if inf_count > 0:
    errors.append(f"{inf_count} Inf values in feature matrix")
if empty_count > 0:
    errors.append(f"{empty_count} empty values in feature matrix")

# ── Feature range check ───────────────────────────────────────
# All normalised ratios should be >= 0
# Ratios over qubit count can exceed 1 but should be < 1000

print("\n  Checking feature value ranges...")
range_errors = 0

for col in FEATURE_COLS:
    vals = [float(r[col]) for r in loaded_rows]
    col_min = min(vals)
    col_max = max(vals)

    if col_min < -1e-6:
        errors.append(
            f"{col}: negative values detected (min={col_min:.4f})"
        )
        range_errors += 1

    if col_max > 10000:
        errors.append(
            f"{col}: suspiciously large values (max={col_max:.4f})"
        )
        range_errors += 1

print(f"  Range errors : {range_errors}  (should be 0)")

# ── Parent linkage check ──────────────────────────────────────
print("\n  Checking parent_sample_id linkage...")

all_sample_ids = {r["sample_id"] for r in loaded_rows}
linkage_errors = 0

for row in loaded_rows:
    if row["label"] == "tampered":
        parent = row["parent_sample_id"]
        if not parent:
            errors.append(
                f"{row['sample_id']}: missing parent_sample_id"
            )
            linkage_errors += 1
        elif parent not in all_sample_ids:
            errors.append(
                f"{row['sample_id']}: parent '{parent}' "
                f"not in dataset"
            )
            linkage_errors += 1
    else:
        if row["parent_sample_id"] not in ("", "None", None):
            errors.append(
                f"{row['sample_id']}: benign circuit has "
                f"non-null parent_sample_id"
            )
            linkage_errors += 1

print(f"  Linkage errors : {linkage_errors}  (should be 0)")

# ── Basic separability check ──────────────────────────────────
# Mean feature values should differ between benign and tampered
# If they are identical something is wrong with the injection

print("\n  Checking benign vs tampered feature separation...")

benign_rows   = [r for r in loaded_rows
                 if r["label"] == "benign"]
tampered_rows = [r for r in loaded_rows
                 if r["label"] == "tampered"]

separation_ok = 0
separation_fail = 0

for col in FEATURE_COLS:
    b_mean = mean([float(r[col]) for r in benign_rows])
    t_mean = mean([float(r[col]) for r in tampered_rows])
    if abs(b_mean - t_mean) > 1e-6:
        separation_ok += 1
    else:
        separation_fail += 1

print(f"  Features with separation : {separation_ok}")
print(f"  Features without         : {separation_fail}")

if separation_fail == len(FEATURE_COLS):
    errors.append(
        "No feature separates benign from tampered. "
        "Check Trojan injection in S2."
    )

# ── Final result ──────────────────────────────────────────────
print("\n" + "=" * 55)
if errors:
    print(f"  ✗ VALIDATION FAILED — {len(errors)} errors")
    for e in errors:
        print(f"    {e}")
    raise RuntimeError("Fix validation errors before S4.")
else:
    print("  ✓ ALL CHECKS PASSED — S3 complete")

# ── Save extractor manifest ───────────────────────────────────

# Compute per-family per-class feature means for manifest
feature_means = {}
for family in FAMILIES:
    feature_means[family] = {}
    for cls in ["benign", "static", "triggered"]:
        cls_rows = [
            r for r in loaded_rows
            if r["algorithm_family"] == family
            and r["trojan_type"] == cls
        ]
        if cls_rows:
            feature_means[family][cls] = {
                col: round(
                    mean([float(r[col]) for r in cls_rows]), 6
                )
                for col in FEATURE_COLS[:5]  # first 5 features
            }

manifest = {
    "extractor_version":  EXTRACTOR_VERSION,
    "qiskit_version":     QISKIT_VERSION,
    "dataset_version":    DATASET_VERSION,
    "features_version":   FEATURES_VERSION,
    "created":            datetime.now(timezone.utc).isoformat(),
    "backend":            BACKEND_NAME,
    "backend_n_qubits":   BACKEND_NQUBITS,
    "transpiler_seed":    TRANSPILER_SEED,
    "opt_levels":         OPT_LEVELS,
    "total_circuits":     len(loaded_rows),
    "total_features":     len(FEATURE_COLS),
    "metadata_columns":   META_COLS,
    "feature_columns":    FEATURE_COLS,
    "label_distribution": {
        k: sum(1 for r in loaded_rows if r["label"] == k)
        for k in ["benign", "tampered"]
    },
    "trojan_type_distribution": {
        k: sum(1 for r in loaded_rows if r["trojan_type"] == k)
        for k in ["benign", "static", "triggered"]
    },
    "family_distribution": {
        f: sum(1 for r in loaded_rows
               if r["algorithm_family"] == f)
        for f in FAMILIES
    },
    "validation_passed":  True,
    "feature_means_sample": feature_means,
}

manifest_path = FEATURES_DIR / "extractor_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n  Manifest saved : {manifest_path}")
print(f"  CSV saved      : {CSV_PATH}")
print()
print(f"  Total circuits : {len(loaded_rows)}")
print(f"  Total features : {len(FEATURE_COLS)}")
print(f"  Backend        : {BACKEND_NAME}")
print(f"  Transpiler seed: {TRANSPILER_SEED}")
print("=" * 55)
print("\nS3 complete. Ready for S4 — split index generation.")


S3 — Final Validation

Reloading features_v1.csv from disk...
  Rows loaded    : 9000
  Columns        : 45

  NaN count   : 0  (should be 0)
  Inf count   : 0  (should be 0)
  Empty count : 0  (should be 0)

  Checking feature value ranges...
  Range errors : 0  (should be 0)

  Checking parent_sample_id linkage...
  Linkage errors : 0  (should be 0)

  Checking benign vs tampered feature separation...
  Features with separation : 39
  Features without         : 0

  ✓ ALL CHECKS PASSED — S3 complete

  Manifest saved : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\features\extractor_manifest.json
  CSV saved      : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\features\features_v1.csv

  Total circuits : 9000
  Total features : 39
  Backend        : FakeGuadalupeV2
  Transpiler seed: 12345

S3 complete. Ready for S4 — split index generation.


In [15]:
import pandas as pd
df = pd.read_csv(FEATURES_DIR / "features_v1.csv")
print(df.shape)
print(df["trojan_type"].value_counts())
print(df[df.columns[6:]].describe().round(3).to_string())

(9000, 45)
trojan_type
benign       3000
static       3000
triggered    3000
Name: count, dtype: int64
       logical_depth_per_qubit  logical_twoq_per_depth  logical_cx_ratio  logical_max_degree_norm  logical_degree_std_over_mean  logical_max_edge_weight_over_total  logical_frac_interactions_top1  logical_frac_interactions_top2  logical_entangling_layers_ratio  logical_fraction_layers_active  logical_max_twoq_in_layer_norm  logical_layer_twoq_std_over_mean  opt0_depth_per_qubit  opt0_twoq_per_depth  opt0_cx_ratio  opt0_max_degree_norm  opt0_degree_std_over_mean  opt0_max_edge_weight_over_total  opt0_frac_interactions_top1  opt0_frac_interactions_top2  opt0_entangling_layers_ratio  opt0_fraction_layers_active  opt0_max_twoq_in_layer_norm  opt0_layer_twoq_std_over_mean  opt3_depth_per_qubit  opt3_twoq_per_depth  opt3_cx_ratio  opt3_max_degree_norm  opt3_degree_std_over_mean  opt3_max_edge_weight_over_total  opt3_frac_interactions_top1  opt3_frac_interactions_top2  opt3_entangling_layers